Set the environment variables.

To securely connect with external APIs (like OpenAI and Tavily), we’ll set up our **environment variables**.  
This ensures our API keys are not hard-coded directly in the notebook, keeping them safe.

- **OPENAI_API_KEY**
- **OPENAI_BASE_URL**
- **TAVILY_API_KEY**

After running this cell, your notebook will be able to access both **OpenAI** and **Tavily APIs** securely.


# Competitor Analysis Using Multi-Agent Retrieval-Augmented Generation (RAG) Collaborative System

## Business Context
Performing competitor analysis with a Multi-Agent Retrieval-Augmented Generation (RAG) Collaborative System.

## Problem Scenario
Conducting a robust competitor analysis of a company is a time-consuming and research-intensive process that may be prone to errors for example, outdated information.

Analysts spend significant time gathering data, identifying relevant competitors, and synthesizing insights into actionable recommendations.

## Objective
To address above mentioned challenges, a Multi-Agent Retrieval-Augmented Generation (RAG) Collaborative System is proposed.

This system will perform competitor analysis of a given company by comparing it with its key rivals in the relevant industry using the latest available web data.

Users will input a company name (e.g., "Tesla") and receive a well-structured report, comparing that company to its primary competitors (e.g., in the electric vehicle sector for Tesla), by leveraging a multi-agent collaborative architecture powered by RAG capabilities.

The system should deliver rapid, accurate, and actionable insights.

## Solution Approach

A Multi-Agent Retrieval-Augmented Generation (RAG) Collaborative System is implemented using a sequential workflow pattern.

This system automates competitor analysis by breaking down the task into distinct steps handled by specialized agents.

The sequential pattern ensures each step builds on the previous one. Also, incorporating RAG to retrieve and augment data from a vector database for context-aware insights.


**Below are the key components of the implementation:**

### State Management

The workflow uses a Pydantic-based state `CompetitiveAnalysisState` to track variables like company name, generated questions, search results, vectorstore status, and the final report.

### Tools
- `suggest_questions`: Generates relevant questions for competitor analysis using the LLM.
- `fetch_search_results`: Searches the web via Tavily API to fetch answers for the questions.
- `store_in_chromadb`: Stores question-answer pairs in ChromaDB for efficient retrieval.
- `generate_report`: Uses RAG to query the vectorstore and draft a structured report.




### Agents

- `Question Generator Agent`: Validates the company, identifies its sector, and generates analysis questions.

- `Data Retrieval and Storage Agent`: Fetches answers from the web and stores them in the vector database.

- `Report Drafter Agent`: Retrieves stored data via RAG and generates a professional, actionable report with sections like Executive Summary, Company Overview, Key Competitors, Strengths/Weaknesses, Market Strategies, and Recommendations.




### Workflow (Sequential Pattern)

The system uses LangGraph's StateGraph to define nodes for each agent.
Edges connect the nodes sequentially as indicated below.

`START → Question Generation → Data Retrieval/Storage → Report Drafting → END.`

The final output is formatted competitive analysis report, displayed in Markdown for readability.

# Installing the Libraries

Installing the libraries:
- **openai==1.99.9** → Official OpenAI client library for interacting with GPT models.  
- **langchain==0.3.27** → Core framework for building applications powered by LLMs.  
- **langchain-openai==0.3.30** → LangChain integration for OpenAI models.  
- **langchain-community==0.3.27** → Community-contributed LangChain modules (tools, integrations).  
- **langgraph==0.6.4** → Build and manage multi-step workflows or agent graphs for LLM-powered systems.  
- **langchain-chroma==0.2.5** → Connector for using **ChromaDB** as a vector database with LangChain.  
- **chromadb==1.0.16** → Open-source vector database for storing and retrieving embeddings.  
- **langchain-tavily==0.2.11** → Integration for the **Tavily API** (specialized web search + retrieval for RAG).  

In [1]:
#Install required packages
!pip install -q openai==1.99.9 \
                langchain==0.3.27 \
                langchain-openai==0.3.30 \
                langchain-community==0.3.27 \
                langgraph==0.6.4 \
                langchain-chroma==0.2.5 \
                chromadb==1.0.16 \
                langchain-tavily==0.2.11

In [2]:

# Verify that the required libraries are installed

import openai
import langchain
import langgraph
import chromadb
import langchain_openai
import langchain_chroma
import langchain_tavily

print("All required libraries imported successfully!")


All required libraries imported successfully!


# Setting up the Environment (5 Marks)

## **Importing the relevant packages**

In [3]:
import os
import json
import random
import datetime

from typing import TypedDict, List, Optional, Dict, Tuple

from pydantic import BaseModel, Field,ConfigDict

import chromadb

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import StructuredTool
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.retrievers import BaseRetriever
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent
from langchain_tavily import TavilySearch

from IPython.display import Image, display, Markdown

from google.colab import userdata

/usr/local/lib/python3.13/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## **Instantiating the required variables (5 Marks)**

Set the environment variables.

To securely connect with external APIs (like OpenAI and Tavily), we’ll set up our **environment variables**.  
This ensures our API keys are not hard-coded directly in the notebook, keeping them safe.

- **OPENAI_API_KEY**
- **OPENAI_BASE_URL**
- **TAVILY_API_KEY**

After running this cell, your notebook will be able to access both **OpenAI** and **Tavily APIs** securely.


In [4]:

# Set environment variables securely
import os
from google.colab import userdata

# Retrieve API keys from Google Colab Secrets
openai_api_key = userdata.get("OPENAI_API_KEY")
tavily_api_key = userdata.get("TAVILY_API_KEY")

# Configure environment variables
os.environ["OPENAI_API_KEY"] = openai_api_key
os.environ["TAVILY_API_KEY"] = tavily_api_key

# Configure the course-provided OpenAI endpoint
os.environ["OPENAI_BASE_URL"] = (
    "https://aibe.mygreatlearning.com/openai/v1"
)

print("Environment variables configured successfully!")


Environment variables configured successfully!


Now that our environment variables are set, let's initialize the **core models** we'll use in this project:

Initialize the Chat Model:

Use ChatOpenAI to create a model instance named llm. You must explicitly set the model to 'gpt-4o-mini', the temperature to 0 (for consistency), and provide the api_key and base_url derived from your environment variables.

In [5]:

# Initialize the OpenAI Chat Model
# Temperature 0 helps maintain consistent responses
# across agents in our multi-agent RAG system.

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"]
)


Initialize the Embedding Model: Create an instance named embedding_model using OpenAIEmbeddings. This model will be responsible for converting your research data into vector representations. Use the 'text-embedding-3-small' model and ensure it points to the correct base_url.

In [6]:

# Initialize the OpenAI Embedding Model
# Ensure the base_url matches the one used for the LLM.

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"]
)


In [7]:
# Test the OpenAI Embedding Model
# Verify that embeddings can be generated successfully

try:
    test_embedding = embedding_model.embed_query(
        "Microsoft Azure cloud computing"
    )

    print("Embedding API connection successful!")
    print("Embedding vector dimensions:", len(test_embedding))

    assert len(test_embedding) == 1536, (
        "Unexpected embedding vector dimensions"
    )

    print("Embedding model verification passed!")

except Exception as e:
    print("Embedding model verification failed.")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

Embedding API connection successful!
Embedding vector dimensions: 1536
Embedding model verification passed!


To enable **Retrieval-Augmented Generation (RAG)**, we need a place to store and query text embeddings.  
Here, we'll use **ChromaDB**, an open-source vector database.

Setting up the Chroma vector database client and a Chroma vector store collection for storing and retrieving search results.

In [8]:

# ChromaDB setup - Persistent client
# Store the vector database on disk

chroma_db_client = chromadb.PersistentClient(
    path="./chroma_db"
)

# Set up vector store
# Store and retrieve competitor research embeddings

vectorstore = Chroma(
    client=chroma_db_client,
    collection_name="competitor_analysis",
    embedding_function=embedding_model
)

print("ChromaDB vector store initialized successfully!")


ChromaDB vector store initialized successfully!


# **State Definitions (6 Marks)**

Lets define the Pydantic models for structured outputs - 'QuestionSuggestion' and the overall state - 'CompetitiveAnalysisState'  .
State Definition ensure type safety and structure the data passed between agents and nodes.

## **QuestionSuggestion (3 Marks)**


This model will help ensure that the responses from the LLM follow a **consistent format**, making them easier to validate and use later.

- Create a class that inherits from `BaseModel`.  
- Fields include:
  - **sector** → The industry sector of the company.  
  - **is_valid_company** →  A boolean flag for whether the company name is recognized.  
  - **questions** → A list of suggested competitive analysis questions.  
  - **error_message** → Optional field to capture any errors that occur.  

- The `Field()` argument ensures that **default values** and **descriptions** are available for each attribute.  


### Define the `QuestionSuggestion` State Model

In this step, you will define a **Pydantic model** to standardize the structured
output returned by the LLM when generating competitive analysis questions.

You need to perform the following steps:

- Create a class named `QuestionSuggestion` that inherits from `BaseModel`.
- Define a field to store the **industry sector** of the company.
- Define a **boolean field** to indicate whether the company name is valid or recognized.
- Define a field to store a **list of competitive analysis questions**.
- Define an **optional field** to capture error messages, if any occur.
- Use `Field()` to:
  - Specify default values where appropriate.
  - Add meaningful descriptions for each field.
- Ensure the model supports structured validation for downstream processing.

Write your code in the below cell.


In [9]:

# Define the QuestionSuggestion Pydantic model
# This model validates the structured output
# generated by the Question Generator Agent.

class QuestionSuggestion(BaseModel):

    sector: str = Field(
        default="Unknown",
        description="Industry sector of the company"
    )

    is_valid_company: bool = Field(
        default=False,
        description="Whether the company is recognized"
    )

    questions: List[str] = Field(
        default_factory=list,
        description="List of competitive analysis questions"
    )

    error_message: Optional[str] = Field(
        default=None,
        description="Error message if question generation fails"
    )



## **Competitive Analysis State (3 marks)**

Next, we’ll create a state object for keeping track off the competitive analysis that keeps track of all the information our pipeline generates and updates during the competitive analysis workflow.

This will act as a “shared memory” object, making it easier to pass data between steps in the system.

- Fields include:
  - **company_name** → The company being analyzed.  
  - **max_num_of_questions** → The maximum number of questions to generate (use a constant as the default).  
  - **sector** → The industry sector (optional).  
  - **is_valid_company** → Boolean flag to mark if the company is recognized (optional).  
  - **question_list** → List of generated analysis questions (optional).  
  - **qna_results** → Stores answers to questions as a list of dictionaries (optional).  
  - **vectorstore** → Chroma vector database instance for retrieval.  
  - **chromadb_insert_status** → Tracks whether data was successfully stored in Chroma (optional).  
  - **report** → Final competitive analysis report (optional).  
  - **error_message** → Captures any error details (optional).

- The `model_config` setting allows **non-Pydantic types** (e.g., Chroma objects).


This state acts as a central memory structure that stores intermediate and
final outputs generated during execution.

Follow the steps below:

- Define a constant to represent the **maximum number of questions** allowed.
- Create a class named `CompetitiveAnalysisState` that inherits from `BaseModel`.
- Define a required field to store the **company name** being analyzed.
- Define a field for the **maximum number of questions**, using the constant as the default value.
- Define optional fields to store:
  - Industry sector
  - Company validity flag
  - Generated question list
  - Question–answer results
  - ChromaDB insertion status
  - Final report
  - Error messages
- Include a field to store the **Chroma vector store instance** used for retrieval.
- Configure the model to allow **non-Pydantic types** using `model_config`.

Write your code in the below cell.

In [10]:

# Set maximum number of competitive research questions
MAX_NUM_OF_QUESTIONS = 40

print("Maximum number of questions:", MAX_NUM_OF_QUESTIONS)


Maximum number of questions: 40


In [11]:
# Define the CompetitiveAnalysisState Pydantic model here

# Define the CompetitiveAnalysisState Pydantic model
# This model maintains shared state across all agents
# in the multi-agent RAG workflow.

class CompetitiveAnalysisState(BaseModel):

    # Allow non-Pydantic objects such as ChromaDB
    model_config = ConfigDict(arbitrary_types_allowed=True)

    # Company being analyzed (required)
    company_name: str

    # Maximum number of research questions
    max_num_of_questions: int = MAX_NUM_OF_QUESTIONS

    # Industry sector of the company
    sector: Optional[str] = None

    # Whether the company is recognized
    is_valid_company: Optional[bool] = None

    # Questions generated by the Question Generator Agent
    question_list: Optional[List[str]] = None

    # Research results retrieved by the Data Retrieval Agent
    qna_results: Optional[List[Dict]] = None

    # ChromaDB vector store used for semantic retrieval
    vectorstore: Optional[Chroma] = None

    # Status of storing research data in ChromaDB
    chromadb_insert_status: Optional[str] = None

    # Final competitor analysis report
    report: Optional[str] = None

    # Error details, if any
    error_message: Optional[str] = None

print("CompetitiveAnalysisState model defined successfully!")


CompetitiveAnalysisState model defined successfully!


# **Tools (13 Marks)**

## **Tavily Search Tool (1 Mark)**

Initialize the Tavily Search tool to retrieve external company-related
information that will later be embedded into the vector store and used
for retrieval-augmented generation (RAG).

The search tool should be configured to:
- Limit the number of returned search results.
- Optionally include a synthesized answer from Tavily.
- Optionally include raw page content for richer context.
- Use an advanced search depth for more comprehensive results.

This tool will be reused by downstream agents for fetching competitive
analysis information.

Write your code in the below cell.


In [12]:

# Initialize the Tavily Search Tool
# Used by agents to retrieve current competitor information.

search = TavilySearch(
    max_results=5,
    include_answer=True,
    include_raw_content=True,
    search_depth="advanced"
)

print("Tavily Search Tool initialized successfully!")


Tavily Search Tool initialized successfully!


Below section defines the tools used by the agents for,
- Suggesting questions,
- Fetching search results,
- Storing data in ChromaDB, and
- Generating the report.

## **Suggest Questions Tool (3 marks)**

Create a custom LangChain tool named `suggest_questions` to generate structured
competitive analysis questions for a given company.

- Define a LangChain tool function that accepts a company name and a maximum
  number of questions as inputs.
- Build an LLM prompt that:
  - Verifies whether the company is real or recognized.
  - Identifies the industry sector of the company.
  - Generates targeted competitor analysis questions covering competitors,
    pricing, technology, supply chain, strengths, weaknesses, and market
    opportunities.
- Invoke the LLM using structured output enforcement with the
  `QuestionSuggestion` schema.
- Extract the sector, company validity flag, and generated questions from the
  structured response.
- Return the extracted data as a JSON-formatted string.
- Handle runtime or model errors by returning default values and an error message.

This tool will be used by downstream agents in the competitive analysis workflow.

Write your code in the below cell.


In [13]:

# Register this function as a LangChain tool
@tool
def suggest_questions(
    company_name: str,
    max_num_of_questions: int
) -> str:
    """
    Generate structured competitive analysis questions
    for a given company using an LLM.
    """

    try:
        # Validate the requested number of questions
        num_questions = max(
            1, min(max_num_of_questions, MAX_NUM_OF_QUESTIONS)
        )

        # Define the prompt for question generation
        prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                """You are an expert competitive intelligence analyst.

                Your responsibilities:
                1. Determine whether the company is real and recognized.
                2. Identify the company's primary industry sector.
                3. Generate strategic competitive analysis questions.
                4. Cover competitors, pricing, technology,
                   supply chain, strengths, weaknesses,
                   market opportunities, and threats.
                5. Generate no more than {max_questions} questions.

                If the company is not recognized:
                - Set is_valid_company to False.
                - Set sector to Unknown.
                - Return an empty questions list.
                - Explain the problem in error_message.

                If the company is recognized:
                - Set is_valid_company to True.
                - Provide its industry sector.
                - Generate relevant research questions.
                - Set error_message to null.

                Return information matching the required schema.
                """
            ),
            (
                "human",
                "Generate competitive analysis questions for "
                "{company_name}."
            )
        ])

        # Enforce structured output using Pydantic
        structured_llm = llm.with_structured_output(
            QuestionSuggestion
        )

        # Create the question generation chain
        chain = prompt | structured_llm

        # Invoke the LLM
        response = chain.invoke({
            "company_name": company_name,
            "max_questions": num_questions
        })

        # Extract the structured response
        result = {
            "sector": response.sector,
            "is_valid_company": response.is_valid_company,
            "questions": response.questions[:num_questions],
            "error_message": response.error_message
        }

        # Return the result as a JSON-formatted string
        return json.dumps(result, indent=2)

    except Exception as e:
        # Handle API, parsing, or runtime errors
        error_result = {
            "sector": "Unknown",
            "is_valid_company": False,
            "questions": [],
            "error_message": str(e)
        }

        return json.dumps(error_result, indent=2)



In [14]:

# Test the Suggest Questions Tool
# Generate 5 competitive analysis questions for Microsoft

test_result = suggest_questions.invoke({
    "company_name": "Microsoft",
    "max_num_of_questions": 5
})

print(test_result)


{
  "sector": "Technology",
  "is_valid_company": true,
  "questions": [
    "Who are Microsoft's main competitors in the cloud computing sector, and how do their offerings compare in terms of pricing and features?",
    "What are the strengths and weaknesses of Microsoft's Office suite compared to competitors like Google Workspace?",
    "How does Microsoft leverage technology advancements, such as AI and machine learning, to enhance its product offerings?",
    "What are the key components of Microsoft's supply chain, and how do they impact the company's ability to deliver products efficiently?",
    "What market opportunities exist for Microsoft in emerging technologies, and what potential threats could hinder its growth?"
  ],
  "error_message": null
}


## **Fetch Search Result Tool (3 Marks)**

The Fetch Search Result tool will take each question, query the **Tavily Search API**, and collect the results.

The function **iterates** through the `question_list` and for each question:
     - Call `search.invoke({"query": question})` to get Tavily search results.  
     - Check if the result contains an `"answer"`.  
     - Store the pair (`question`, `answer`) in a list of dictionaries.  
Additionally, the function can:
- **Handle missing answers** by storing an empty string.
- Handle errors** with a `try/except` block and log them for debugging.  
- Return the **final results as a JSON string**.

Create a custom LangChain tool named `fetch_search_results` to retrieve answers
for generated competitive analysis questions using the Tavily Search API.

In this cell, implement logic to:
- Define a LangChain tool function that accepts a list of questions as input.
- Iterate over each question in the provided list.
- Query the Tavily Search API for each question.
- Check whether the search response contains a valid answer.
- Store each question and its corresponding answer as a dictionary.
- Handle missing or empty answers by storing an empty string.
- Catch and log errors encountered during search execution.
- Return the complete list of question–answer pairs as a JSON-formatted string.

This tool will be used by downstream nodes to enrich the pipeline with
external search-based information.

Write your code in the below cell.


In [15]:

# Register this function as a LangChain tool
@tool
def fetch_search_results(question_list: List[str]) -> str:
    """
    Fetch answers for competitive analysis questions
    using the Tavily Search API.

    Returns a JSON string containing question-answer pairs.
    """

    # Store the results for all questions
    search_results = []

    # Process each research question
    for question in question_list:
        try:
            # Search the web using Tavily
            response = search.invoke({
                "query": question
            })

            # Extract the answer if available
            answer = ""

            if isinstance(response, dict):
                answer = response.get("answer") or ""

            elif isinstance(response, str):
                # Handle JSON-formatted string responses
                try:
                    parsed_response = json.loads(response)

                    if isinstance(parsed_response, dict):
                        answer = parsed_response.get("answer") or ""

                except json.JSONDecodeError:
                    answer = ""

            # Store the question-answer pair
            search_results.append({
                "question": question,
                "answer": answer
            })

        except Exception as e:
            # Log the error for troubleshooting
            print(f"Search failed for question: {question}")
            print(f"Error: {type(e).__name__}: {e}")

            # Preserve the question with an empty answer
            search_results.append({
                "question": question,
                "answer": ""
            })

    # Return all question-answer pairs as JSON
    return json.dumps(search_results, indent=2)



In [16]:

# Test the Fetch Search Results Tool
# Retrieve competitor information using Tavily

test_questions = [
    "Who are Microsoft's main competitors in cloud computing?"
]

# Invoke the LangChain tool
search_test_result = fetch_search_results.invoke({
    "question_list": test_questions
})

# Display the retrieved results
print(search_test_result)


[
  {
    "question": "Who are Microsoft's main competitors in cloud computing?",
    "answer": "According to the sources, Microsoft's main competitors in cloud computing are Amazon Web Services (AWS) and Google Cloud. The sources also mention Oracle and IBM as significant competitors in the cloud computing space, often referred to as the \"Big Three\" alongside Azure. Gartner lists AWS, Oracle, Google, IBM, SAP, Broadcom (VMware), Rackspace Technology, and Teradata as top alternatives to Microsoft in cloud computing. Ultima Markets highlights AWS as the number one contender to Microsoft in the global cloud computing market due to its scale, breadth of infrastructure services, and extensive enterprise adoption."
  }
]


## **Store in ChromaDB Tool (4 Marks)**

After fetching question-answer pairs, they must be stored in **ChromaDB** so they can later be retrieved during the analysis.  
This tool will embed the answers and insert them into the vector store along with metadata for traceability.

The function `store_in_chromadb`:
- Loop through each question–answer dictionary.  
- **Skip empty answers** (don’t embed/store them).  
- For valid answers:  
    - Create a `documents` list containing the answer text.  
    - Create `metadatas` with fields such as:  
        - `question` → the original question  
        - `source_type` → `"answer"`  
        - `search_success` → whether a valid answer was found  
    - Create unique `ids` for each entry (e.g., `"doc_1"`, `"doc_2"`).  
- Use `vectorstore.add_texts()` to insert documents, metadata, and IDs into ChromaDB.  

Create a custom LangChain tool named `store_in_chromadb` to persist question–answer
data into a ChromaDB vector store for later retrieval.

In this cell, implement logic to:
- Define a LangChain tool function that accepts a list of question–answer
  dictionaries as input.
- Iterate through the list of question–answer entries.
- Skip entries where the answer is empty or missing.
- For each valid answer:
  - Prepare a list of documents containing the answer text.
  - Attach metadata that includes the original question, the source type, and
    whether the search was successful.
  - Generate unique document IDs for insertion into the vector store.
- Insert documents, metadata, and IDs into ChromaDB using the vector store API.
- Track whether all valid answers were successfully stored.
- Return the final insertion status as a JSON-formatted string.

This tool enables retrieval-augmented generation in later stages of the workflow.

Write your code in the below cell.


In [17]:

# Register this function as a LangChain tool
@tool
def store_in_chromadb(qna_results: List[Dict]) -> str:
    """
    Store valid question-answer results in ChromaDB
    for retrieval-augmented competitor analysis.

    Returns a JSON string describing storage status.
    """

    documents = []
    metadatas = []
    ids = []

    # Track valid and skipped answers
    skipped_count = 0

    try:
        # Process each question-answer pair
        for item in qna_results:

            question = str(item.get("question") or "")
            answer = str(item.get("answer") or "").strip()

            # Skip missing or empty answers
            if not answer:
                skipped_count += 1
                continue

            # Store the answer as a document
            documents.append(answer)

            # Attach metadata for traceability
            metadatas.append({
                "question": question,
                "source_type": "answer",
                "search_success": True
            })

            # Generate a unique document ID
            import uuid
            document_id = f"doc_{uuid.uuid4().hex}"
            ids.append(document_id)

        # Insert valid documents into ChromaDB
        if documents:
            vectorstore.add_texts(
                texts=documents,
                metadatas=metadatas,
                ids=ids
            )

        # Return storage status
        result = {
            "status": "success",
            "stored_count": len(documents),
            "skipped_count": skipped_count,
            "message": (
                f"Successfully stored {len(documents)} "
                "documents in ChromaDB."
            )
        }

        return json.dumps(result, indent=2)

    except Exception as e:
        # Capture database or embedding API errors
        error_result = {
            "status": "error",
            "stored_count": 0,
            "skipped_count": skipped_count,
            "message": str(e)
        }

        return json.dumps(error_result, indent=2)


In [18]:

# Test the Store in ChromaDB Tool
# Use the question-answer results retrieved from Tavily

# Convert the previous JSON response into a Python list
qna_data = json.loads(search_test_result)

# Store the research results in ChromaDB
storage_test_result = store_in_chromadb.invoke({
    "qna_results": qna_data
})

# Display the storage status
print(storage_test_result)


{
  "status": "success",
  "stored_count": 1,
  "skipped_count": 0,
  "message": "Successfully stored 1 documents in ChromaDB."
}


In [19]:
# Test ChromaDB Semantic Search
# Verify retrieval of stored competitor research

query = "Who are Microsoft's main cloud competitors?"

try:
    retrieved_docs = vectorstore.similarity_search(
        query,
        k=1
    )

    print("Number of documents retrieved:", len(retrieved_docs))

    if retrieved_docs:
        print("\nChromaDB semantic search successful!")

        for doc in retrieved_docs:
            print("\n--- Retrieved Document Preview ---")
            print(doc.page_content[:500])

            print("\n--- Document Metadata ---")
            print(doc.metadata)

    else:
        print("\nNo documents found in ChromaDB.")
        print("Run the Data Retrieval and Storage Node first.")

except Exception as e:
    print("ChromaDB semantic search failed.")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

Number of documents retrieved: 1

ChromaDB semantic search successful!

--- Retrieved Document Preview ---
According to the sources, Microsoft's main competitors in cloud computing are Amazon Web Services (AWS) and Google Cloud. The sources also mention Oracle and IBM as significant competitors in the cloud computing space, often referred to as the "Big Three" alongside Azure. Gartner lists AWS, Oracle, Google, IBM, SAP, Broadcom (VMware), Rackspace Technology, and Teradata as top alternatives to Microsoft in cloud computing. Ultima Markets highlights AWS as the number one contender to Microsoft in t

--- Document Metadata ---
{'question': "Who are Microsoft's main competitors in cloud computing?", 'source_type': 'answer', 'search_success': True}


## **Generate Report Tool (2 Marks)**

The `generate_report` tool will be used to define precisely how to use retrieved context for generating the report by
- using the retrieved context as the primary source and comparing competitors across dimensions (market share, products, pricing, tech, supply chain, customer sentiment).
- highlighting differentiators and actionable strategies.
- structuring the final output into the named sections and citing context where relevant.

The functions builds a `human` message that requests a report for the given `company_name` and `sector` and includes a `{context}` placeholder which will be filled at runtime with retrieved documents from the vectorstore and combine messages into a `ChatPromptTemplate.from_messages([...])`

Create a custom LangChain tool named `generate_report` to define how retrieved
context is used to generate the final competitive analysis report.

In this cell, implement logic to:
- Define a LangChain tool function that accepts the company name and industry
  sector as inputs.
- Construct a system message that instructs the LLM to perform competitor
  analysis using retrieved context as the primary source of information.
- Ensure the analysis compares competitors across dimensions such as market
  share, products, pricing, technology, supply chain, and customer sentiment.
- Include instructions to highlight differentiators and propose actionable
  strategies.
- Build a human message that requests report generation for the given company
  and sector and includes a `{context}` placeholder.
- Combine system and human messages into a `ChatPromptTemplate`.
- Return the prompt template for downstream invocation (do not execute the LLM
  in this tool).

This tool defines the report structure and is used later within a RAG workflow.

Write your code in the below cell.


In [20]:

# Define the Generate Report Tool
# Creates a prompt template for the Report Drafter Agent.
# The LLM will be invoked later in the workflow.

def generate_report(
    company_name: str,
    sector: str
) -> ChatPromptTemplate:
    """
    Create a competitive analysis report prompt.

    The retrieved context will be provided later
    by the Report Drafter Agent.
    """

    # System instructions for the report
    system_message = """
    You are an expert competitive intelligence analyst.

    Your task is to generate a comprehensive competitor
    analysis report using the retrieved context as
    the primary source of information.

    Follow these guidelines:

    1. Analyze the company's competitive position
       within its industry sector.

    2. Identify key competitors and compare them across:
       - Market share and market positioning
       - Products and services
       - Pricing strategies
       - Technology and innovation
       - Supply chain capabilities
       - Customer sentiment and satisfaction

    3. Highlight competitive differentiators,
       strengths, weaknesses, opportunities, and threats.

    4. Provide actionable business recommendations
       and strategies to improve competitiveness.

    5. Structure the report using these sections:
       - Executive Summary
       - Company Overview
       - Industry and Market Overview
       - Key Competitors
       - Competitive Comparison
       - SWOT Analysis
       - Strategic Recommendations
       - Conclusion

    6. Base factual claims on the retrieved context.
       Do not invent market share figures, pricing,
       financial data, or unsupported facts.

    7. Cite supporting context where relevant.
       If information is unavailable, clearly
       identify the evidence gap.

    Produce a professional, structured report
    suitable for business decision-makers.
    """

    # Human message containing company, sector,
    # and the context placeholder
    human_message = """
    Generate a detailed competitive analysis report
    for the following company:

    Company Name: {company_name}
    Industry Sector: {sector}

    Retrieved Research Context:
    {context}

    Use the retrieved information to compare the
    company against its major competitors.

    Provide meaningful insights and actionable
    recommendations.
    """

    # Combine system and human messages
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_message),
        ("human", human_message)
    ])

    # Pre-fill company and sector while keeping
    # context available for downstream retrieval.
    prompt = prompt.partial(
        company_name=company_name,
        sector=sector
    )

    return prompt


print("Generate Report Tool defined successfully!")



Generate Report Tool defined successfully!


In [21]:

# Test the Generate Report function
# Verify that the prompt template is created correctly

report_prompt = generate_report(
    company_name="Microsoft",
    sector="Technology"
)

# Verify the returned object
print("Prompt type:", type(report_prompt).__name__)

# Verify the required runtime variables
print("Required input variables:", report_prompt.input_variables)

# Insert sample retrieved context
formatted_prompt = report_prompt.format_messages(
    context="Microsoft Azure competes with AWS and Google Cloud."
)

print("\nPrompt created successfully!")
print("Number of messages:", len(formatted_prompt))


Prompt type: ChatPromptTemplate
Required input variables: ['context']

Prompt created successfully!
Number of messages: 2


# **Agents (20 Marks)**

## **Initialize the Agents (2 Marks)**

Below section creates reactive agents using LangChain's create_react_agent. Each agent is assigned specific tools:
- question_generator_agent: Uses suggest_questions tool.
- data_retrieval_storage_agent: Uses fetch_search_results and store_in_chromadb tool.
- report_drafter_agent: Uses generate_report tool.

### Questions Generator Agent

In [22]:

# Initialize the Questions Generator Agent
# This agent uses the suggest_questions tool
# to generate structured competitor analysis questions.

question_generator_agent = create_react_agent(
    model=llm,
    tools=[suggest_questions]
)

print("Questions Generator Agent initialized successfully!")


Questions Generator Agent initialized successfully!


### Data Retrieval & Storage Agent

In [23]:

# Initialize the Data Retrieval & Storage Agent
# This agent retrieves competitor research using Tavily
# and stores the results in ChromaDB for RAG.

data_retrieval_storage_agent = create_react_agent(
    model=llm,
    tools=[
        fetch_search_results,
        store_in_chromadb
    ]
)

print("Data Retrieval & Storage Agent initialized successfully!")


Data Retrieval & Storage Agent initialized successfully!


### Report Drafter Agent

In [24]:

# Initialize the Report Drafter Agent
# Register an adapter for the existing generate_report function.

@tool
def generate_report_tool(company_name: str, sector: str) -> str:
    """
    Create a competitive analysis report prompt template
    for the specified company and industry sector.
    """
    prompt = generate_report(
        company_name=company_name,
        sector=sector
    )

    # Return a serializable representation of the prompt.
    # The Report Drafter Node will use the original
    # generate_report function with retrieved context.
    return str(prompt)


# Create the Report Drafter Agent
report_drafter_agent = create_react_agent(
    model=llm,
    tools=[generate_report_tool]
)

print("Report Drafter Agent initialized successfully!")


Report Drafter Agent initialized successfully!


# **Defining the Nodes (18 Marks)**

## **Question Generator Node (6 Marks)**

Question Generator Node - Generates questions and updates state.

The run_question_generator node invokes the question-generation agent to produce structured JSON output and then parses that output to update the pipeline state by:
- invoking the agent with a short system+human message (company name, max questions) and retrieving the agent response.
- searching responses in reverse for the last JSON block (to find the structured payload) and using json.loads to parse it.
- updating CompetitiveAnalysisState fields: sector, is_valid_company, question_list, and error_message.
- handling errors and parsing failures gracefully by logging the issue and setting safe defaults (sector="", is_valid_company=False, question_list=[], error_message=<error>).
- returning the updated state for downstream nodes to consume.

Implement the `run_question_generator` node to generate competitive analysis
questions and update the shared pipeline state.

In this cell, implement logic to:
- Define a node function that accepts a `CompetitiveAnalysisState` object.
- Create a short system message and a human message containing the company name
  and maximum number of questions.
- Invoke the question-generation agent with the constructed messages.
- Inspect the agent response and search messages in reverse order to locate the
  last valid JSON payload.
- Parse the JSON content to extract sector, company validity, generated
  questions, and any error message.
- Update the corresponding fields in the `CompetitiveAnalysisState`.
- Handle JSON parsing errors and agent invocation failures gracefully by setting
  safe default values for state fields.
- Return the updated state so it can be consumed by downstream nodes.

This node initializes the pipeline by producing structured inputs for
subsequent retrieval and analysis steps.

Write your code in the below cell.


In [25]:

# Question Generator Node
# Invokes the Questions Generator Agent and updates
# the shared CompetitiveAnalysisState.

def run_question_generator(
    state: CompetitiveAnalysisState
) -> CompetitiveAnalysisState:

    # Initialize safe defaults
    state.sector = ""
    state.is_valid_company = False
    state.question_list = []
    state.error_message = None

    try:
        # Read company details from shared state
        company_name = state.company_name
        max_questions = state.max_num_of_questions

        # System instructions for the agent
        system_message = SystemMessage(
            content=(
                "You are a competitive intelligence research assistant. "
                "Use the suggest_questions tool to verify the company, "
                "identify its industry sector, and generate structured "
                "competitive analysis questions. "
                "Return the tool's JSON result without changing its fields."
            )
        )

        # Human message with company information
        human_message = HumanMessage(
            content=(
                f"Company name: {company_name}\n"
                f"Maximum number of questions: {max_questions}\n"
                "Call the suggest_questions tool and return its JSON result."
            )
        )

        # Invoke the Questions Generator Agent
        response = question_generator_agent.invoke({
            "messages": [system_message, human_message]
        })

        # Search agent messages in reverse order
        # to locate the latest valid JSON response
        parsed_result = None

        for message in reversed(response.get("messages", [])):

            content = message.content

            # Convert non-string content when necessary
            if not isinstance(content, str):
                continue

            # First attempt: parse the complete message
            try:
                candidate = json.loads(content)
            except json.JSONDecodeError:
                # Handle JSON embedded in Markdown fences
                cleaned = content.strip()

                if "```" in cleaned:
                    cleaned = cleaned.replace("```json", "")
                    cleaned = cleaned.replace("```", "").strip()

                try:
                    candidate = json.loads(cleaned)
                except json.JSONDecodeError:
                    continue

            # Validate expected structured fields
            if isinstance(candidate, dict) and all(
                key in candidate
                for key in ("sector", "is_valid_company", "questions")
            ):
                parsed_result = candidate
                break

        # Update state with the parsed JSON
        if parsed_result is not None:

            state.sector = parsed_result.get("sector", "")
            state.is_valid_company = parsed_result.get(
                "is_valid_company", False
            )
            state.question_list = parsed_result.get(
                "questions", []
            )
            state.error_message = parsed_result.get(
                "error_message"
            )

        else:
            raise ValueError(
                "No valid question-generation JSON found in agent response."
            )

    except Exception as e:
        # Handle agent execution and parsing errors
        print(f"Question Generator Node Error: {e}")

        state.sector = ""
        state.is_valid_company = False
        state.question_list = []
        state.error_message = str(e)

    # Return updated shared state
    return state


print("Question Generator Node defined successfully!")



Question Generator Node defined successfully!


In [26]:

# Test the Question Generator Node
# Verify that the agent updates CompetitiveAnalysisState

# Initialize the state with a test company
test_state = CompetitiveAnalysisState(
    company_name="Microsoft",
    max_num_of_questions=5
)

# Execute the Question Generator Node
updated_state = run_question_generator(test_state)

# Display the updated state
print("Company:", updated_state.company_name)
print("Sector:", updated_state.sector)
print("Valid Company:", updated_state.is_valid_company)
print("Number of Questions:", len(updated_state.question_list))
print("Error Message:", updated_state.error_message)

# Display generated research questions
print("\nGenerated Competitive Analysis Questions:")

for index, question in enumerate(
    updated_state.question_list, start=1
):
    print(f"{index}. {question}")


Company: Microsoft
Sector: Technology
Valid Company: True
Number of Questions: 5
Error Message: None

Generated Competitive Analysis Questions:
1. Who are Microsoft's main competitors in the cloud computing sector, and how do their offerings compare in terms of pricing and features?
2. What are the strengths and weaknesses of Microsoft's Office suite compared to competitors like Google Workspace?
3. How does Microsoft leverage technology advancements, such as AI and machine learning, to enhance its product offerings?
4. What are the key components of Microsoft's supply chain, and how do they impact the company's ability to deliver products efficiently?
5. What market opportunities exist for Microsoft in emerging technologies, and what potential threats could hinder its growth?


## **Data Retrieval and Storage Node (6 Marks)**

This Node fetches and stores the search results.

The run_data_retrieval_storage node fetches answers for generated questions and stores them in ChromaDB, then updates the pipeline state by:
- verifying questions exist and short-circuiting with an error if none were generated.
- invoking the data_retrieval_storage agent with a short system+human message (questions) and retrieving the agent response.
- parsing the last message as JSON to populate state.qna_results (the list of question–answer dicts).
- extracting the `store_in_chromadb` tool call output from `response["messages"][-1].tool_calls` (if present) to set state.chromadb_insert_status, with a fallback to read `chromadb_insert_status` from the parsed last message.
handling JSON parse errors by setting state.error_message to a descriptive message.
- returning the updated CompetitiveAnalysisState for downstream nodes to consume.


Implement the `run_data_retrieval_storage` node to fetch external search results
and persist them in the vector database while updating the pipeline state.

In this cell, implement logic to:
- Define a node function that accepts a `CompetitiveAnalysisState` object.
- Verify that a list of generated questions exists and terminate early with an
  error message if no questions are available.
- Create a short system message and a human message containing the list of
  generated questions.
- Invoke the data retrieval and storage agent with the constructed messages.
- Extract the final response content and parse it as JSON to obtain the
  question–answer results.
- Update the state with the parsed question–answer data.
- Inspect the agent response for tool calls and extract the
  `chromadb_insert_status` from the `store_in_chromadb` tool invocation when
  present.
- Fall back to extracting the ChromaDB insertion status from the parsed response
  when tool calls are not available.
- Handle JSON parsing errors by setting a descriptive error message in the
  pipeline state.
- Return the updated state for downstream nodes to consume.

This node enriches the pipeline with external knowledge and ensures data is
persisted for retrieval-augmented generation.

Write your code in the below cell.


In [27]:

# Data Retrieval and Storage Node
# Fetches competitive research using Tavily,
# stores the answers in ChromaDB, and updates state.

def run_data_retrieval_storage(
    state: CompetitiveAnalysisState
) -> CompetitiveAnalysisState:

    # Initialize safe defaults
    state.qna_results = []
    state.chromadb_insert_status = None
    state.error_message = None

    # Check whether research questions are available
    if not state.question_list:
        state.error_message = (
            "No research questions available for data retrieval."
        )
        print(state.error_message)
        return state

    try:
        # System instructions for the agent
        system_message = SystemMessage(
            content=(
                "You are a competitive intelligence research agent. "
                "You must perform the following steps in order:\n"
                "1. Call fetch_search_results using all provided questions.\n"
                "2. Parse the returned JSON question-answer pairs.\n"
                "3. Call store_in_chromadb with those pairs.\n"
                "4. Return the question-answer JSON as your final response.\n"
                "Do not invent research findings."
            )
        )

        # Provide the generated questions to the agent
        human_message = HumanMessage(
            content=(
                "Research the following competitive analysis questions "
                "and store the results in ChromaDB:\n"
                f"{json.dumps(state.question_list, indent=2)}"
            )
        )

        # Invoke the Data Retrieval and Storage Agent
        response = data_retrieval_storage_agent.invoke({
            "messages": [system_message, human_message]
        })

        messages = response.get("messages", [])

        # Helper: parse JSON from an agent message
        def parse_json(content):
            if not isinstance(content, str):
                return None

            text = content.strip()

            try:
                return json.loads(text)
            except json.JSONDecodeError:
                pass

            # Handle JSON enclosed in Markdown fences
            if "```" in text:
                import re

                blocks = re.findall(
                    r"```(?:json)?\s*(.*?)```",
                    text,
                    flags=re.DOTALL
                )

                for block in blocks:
                    try:
                        return json.loads(block.strip())
                    except json.JSONDecodeError:
                        continue

            return None

        # Locate the question-answer results
        # and ChromaDB insertion status in tool messages
        for message in messages:
            tool_name = getattr(message, "name", "")
            parsed = parse_json(message.content)

            if parsed is None:
                continue

            if tool_name == "fetch_search_results":
                if isinstance(parsed, list):
                    state.qna_results = parsed

            elif tool_name == "store_in_chromadb":
                if isinstance(parsed, dict):
                    state.chromadb_insert_status = json.dumps(parsed)

        # Fallback: inspect messages in reverse order
        # if tool messages were not available
        if not state.qna_results:
            for message in reversed(messages):
                parsed = parse_json(message.content)

                if isinstance(parsed, list):
                    if all(
                        isinstance(item, dict)
                        and "question" in item
                        and "answer" in item
                        for item in parsed
                    ):
                        state.qna_results = parsed
                        break

        # Fallback for ChromaDB status
        if state.chromadb_insert_status is None:
            for message in reversed(messages):
                parsed = parse_json(message.content)

                if isinstance(parsed, dict):
                    if "stored_count" in parsed:
                        state.chromadb_insert_status = json.dumps(parsed)
                        break

        # Validate the retrieval and storage results
        if not state.qna_results:
            raise ValueError(
                "No valid question-answer results found."
            )

        if state.chromadb_insert_status is None:
            raise ValueError(
                "ChromaDB insertion status was not returned."
            )

        # Check whether ChromaDB reported an error
        storage_status = json.loads(
            state.chromadb_insert_status
        )

        if storage_status.get("status") != "success":
            raise RuntimeError(
                storage_status.get(
                    "message",
                    "ChromaDB storage failed."
                )
            )

        print("Data retrieval and storage completed successfully!")
        print("Questions processed:", len(state.qna_results))
        print(
            "Documents stored:",
            storage_status.get("stored_count", 0)
        )

    except Exception as e:
        # Preserve the error for downstream nodes
        state.error_message = str(e)

        print(
            "Data Retrieval and Storage Node Error:",
            state.error_message
        )

    # Return the updated shared state
    return state


print("Data Retrieval and Storage Node defined successfully!")




Data Retrieval and Storage Node defined successfully!


In [28]:

# Test the Data Retrieval and Storage Node
# Use the state generated by the Question Generator Node

print("Starting Data Retrieval and Storage Node test...")

# Verify that research questions are available
print("Company:", updated_state.company_name)
print("Questions available:", len(updated_state.question_list))

# Execute the Data Retrieval and Storage Node
retrieval_state = run_data_retrieval_storage(updated_state)

# Display the updated state
print("\n--- Data Retrieval Results ---")
print("Company:", retrieval_state.company_name)
print("Number of Q&A results:", len(retrieval_state.qna_results))
print("ChromaDB Insert Status:",
      retrieval_state.chromadb_insert_status)
print("Error Message:", retrieval_state.error_message)

# Display a preview of retrieved research
if retrieval_state.qna_results:
    print("\n--- First Research Result ---")
    print(json.dumps(
        retrieval_state.qna_results[0],
        indent=2
    ))


Starting Data Retrieval and Storage Node test...
Company: Microsoft
Questions available: 5
Data retrieval and storage completed successfully!
Questions processed: 5
Documents stored: 5

--- Data Retrieval Results ---
Company: Microsoft
Number of Q&A results: 5
ChromaDB Insert Status: {"status": "success", "stored_count": 5, "skipped_count": 0, "message": "Successfully stored 5 documents in ChromaDB."}
Error Message: None

--- First Research Result ---
{
  "question": "Who are Microsoft's main competitors in the cloud computing sector, and how do their offerings compare in terms of pricing and features?",
  "answer": "Microsoft's main competitors in the cloud computing sector are Amazon Web Services (AWS), Google Cloud, and Oracle. AWS and Google Cloud are the closest rivals to Microsoft's Azure, competing closely in terms of market share and innovation. AWS holds 28% of the global cloud infrastructure market, followed by Azure at 21%, and Google Cloud at 14% according to Synergy Resear

## **Report Drafter Node (6 Marks)**

Below Node generates the report leveraging RAG and prints a preview.

The run_report_drafter node builds a RAG-powered report by retrieving stored Q&A context and invoking the report-generation prompt, then updates the pipeline state by:
- creating a retriever from state.vectorstore (search_kwargs={"k": 10}) and querying it for competitive-analysis documents for the company.
- handling the "no documents" case by setting state.report and state.error_message and short-circuiting.
- concatenating retrieved documents' `page_content` into a single `context` string for the LLM.
- invoking the `generate_report` tool to obtain a `ChatPromptTemplate` (expects a prompt template, not a final string).
- validating the returned object is a `ChatPromptTemplate`; on mismatch, set `state.error_message` and `state.report` appropriately and return.
- composing the prompt template with the LLM (e.g., `chain = prompt_template | llm`) and invoking the chain with `{"context": context}` to produce the report.
storing the generated report in `state.report`, logging a brief preview, and handling any exceptions by setting `state.error_message` and a failure `state.report`.
- returning the updated CompetitiveAnalysisState for downstream consumption.


Implement the `run_report_drafter` node to generate the final competitive analysis
report using retrieval-augmented generation (RAG).

In this cell, implement logic to:
- Define a node function that accepts a `CompetitiveAnalysisState` object.
- Create a retriever from the vector store using an appropriate search
  configuration.
- Query the retriever for competitive analysis documents related to the company.
- Detect the case where no documents are retrieved and update the state with a
  descriptive error message and report.
- Combine the retrieved document contents into a single context string.
- Invoke the report-generation tool to obtain a prompt template.
- Validate that the returned object is a prompt template and not a final string.
- Compose the prompt template with the LLM and invoke it using the retrieved
  context.
- Store the generated report in the pipeline state and log a short preview.
- Handle unexpected errors by updating the state with failure information.
- Return the updated state for downstream consumption.

This node produces the final report output of the competitive analysis workflow.

Write your code in the below cell.


In [29]:

# Report Drafter Node
# Retrieves research from ChromaDB and generates
# a RAG-based competitive analysis report.

def run_report_drafter(
    state: CompetitiveAnalysisState
) -> CompetitiveAnalysisState:

    # Initialize report fields
    state.report = None
    state.error_message = None

    try:
        # Step 1: Get the ChromaDB vector store
        if state.vectorstore is None:
            state.vectorstore = vectorstore

        # Step 2: Create a retriever
        retriever = state.vectorstore.as_retriever(
            search_kwargs={"k": 10}
        )

        # Step 3: Retrieve relevant research documents
        query = (
            f"Competitive analysis of {state.company_name} "
            f"in the {state.sector or 'technology'} sector. "
            "Competitors, pricing, products, technology, "
            "market share, strengths, weaknesses, "
            "supply chain and opportunities."
        )

        retrieved_docs = retriever.invoke(query)

        # Step 4: Handle missing documents
        if not retrieved_docs:
            state.report = (
                "Report generation failed: "
                "No relevant research documents found."
            )
            state.error_message = (
                "No documents retrieved from ChromaDB."
            )
            return state

        # Step 5: Combine retrieved documents
        context = "\n\n".join(
            f"Research Document {i}:\n"
            f"Question: {doc.metadata.get('question', 'N/A')}\n"
            f"Answer: {doc.page_content}"
            for i, doc in enumerate(retrieved_docs, start=1)
        )

        # Step 6: Generate the report prompt template
        prompt_template = generate_report(
            company_name=state.company_name,
            sector=state.sector or "Unknown"
        )

        # Step 7: Validate the prompt template
        if not isinstance(prompt_template, ChatPromptTemplate):
            raise TypeError(
                "generate_report must return a ChatPromptTemplate."
            )

        # Step 8: Combine the prompt template with the LLM
        chain = prompt_template | llm

        # Step 9: Generate the competitive analysis report
        response = chain.invoke({
            "context": context
        })

        # Step 10: Extract the generated report
        if isinstance(response.content, str):
            report_text = response.content
        else:
            report_text = str(response.content)

        if not report_text.strip():
            raise ValueError(
                "The LLM returned an empty report."
            )

        # Step 11: Update the shared state
        state.report = report_text
        state.error_message = None

        # Step 12: Display report preview
        print("Report generated successfully!")
        print("Documents retrieved:", len(retrieved_docs))
        print("\n--- Report Preview ---")
        print(report_text[:1500])

    except Exception as e:
        # Handle retrieval or report-generation errors
        state.error_message = str(e)
        state.report = (
            f"Report generation failed: {e}"
        )

        print("Report Drafter Node Error:", e)

    # Return updated state
    return state


print("Report Drafter Node defined successfully!")



Report Drafter Node defined successfully!


In [30]:

# Test the Report Drafter Node
# Generate a competitive analysis report using RAG

print("Starting Report Drafter Node test...")

# Use the state from the previous retrieval test
print("Company:", retrieval_state.company_name)
print("Sector:", retrieval_state.sector)

# Execute the Report Drafter Node
report_state = run_report_drafter(retrieval_state)

# Display the results
print("\n--- Report Drafter Test Results ---")
print("Company:", report_state.company_name)
print("Report Generated:", bool(
    report_state.report and
    not report_state.error_message
))
print("Error Message:", report_state.error_message)

# Display the complete report
if report_state.report and not report_state.error_message:
    print("\n--- Competitive Analysis Report ---")
    display(Markdown(report_state.report))


Starting Report Drafter Node test...
Company: Microsoft
Sector: Technology
Report generated successfully!
Documents retrieved: 10

--- Report Preview ---
# Competitive Analysis Report for Microsoft

## Executive Summary
Microsoft operates in the highly competitive technology sector, particularly focusing on cloud computing and emerging technologies such as artificial intelligence (AI). This report analyzes Microsoft's competitive position, identifies key competitors, and provides a comprehensive comparison across various dimensions. The analysis highlights Microsoft's strengths in AI integration and strategic alliances while also addressing potential threats from competition and regulatory challenges. Actionable recommendations are provided to enhance Microsoft's competitiveness in the market.

## Company Overview
Microsoft Corporation is a leading global technology company known for its software products, cloud services, and hardware solutions. Key offerings include the Windows operat

# Competitive Analysis Report for Microsoft

## Executive Summary
Microsoft operates in the highly competitive technology sector, particularly focusing on cloud computing and emerging technologies such as artificial intelligence (AI). This report analyzes Microsoft's competitive position, identifies key competitors, and provides a comprehensive comparison across various dimensions. The analysis highlights Microsoft's strengths in AI integration and strategic alliances while also addressing potential threats from competition and regulatory challenges. Actionable recommendations are provided to enhance Microsoft's competitiveness in the market.

## Company Overview
Microsoft Corporation is a leading global technology company known for its software products, cloud services, and hardware solutions. Key offerings include the Windows operating system, Microsoft Office suite, Azure cloud services, and various enterprise solutions. The company has made significant investments in AI and cloud technologies, positioning itself as a key player in these rapidly evolving markets.

## Industry and Market Overview
The technology sector is characterized by rapid innovation and intense competition, particularly in cloud computing and AI. The global cloud infrastructure market is dominated by a few major players, with Amazon Web Services (AWS), Google Cloud, and Oracle being the primary competitors to Microsoft Azure. The increasing demand for cloud services and AI integration presents significant growth opportunities for Microsoft, but also poses challenges from competitors and regulatory scrutiny.

## Key Competitors
1. **Amazon Web Services (AWS)**
   - Market Share: 28% of global cloud infrastructure market.
   - Strengths: Extensive service offerings, strong enterprise adoption, and a robust infrastructure.
   - Weaknesses: Higher pricing for certain services compared to competitors.

2. **Google Cloud**
   - Market Share: 14% of global cloud infrastructure market.
   - Strengths: Strong capabilities in data analytics and machine learning.
   - Weaknesses: Limited enterprise adoption compared to AWS and Azure.

3. **Oracle**
   - Market Share: Not specified, but recognized as a significant player in cloud infrastructure and enterprise software.
   - Strengths: Comprehensive suite of cloud services and enterprise resource planning solutions.
   - Weaknesses: Perceived as less innovative compared to AWS and Azure.

## Competitive Comparison
| Feature/Aspect          | Microsoft Azure       | Amazon Web Services (AWS) | Google Cloud          | Oracle Cloud         |
|------------------------|-----------------------|---------------------------|-----------------------|----------------------|
| **Market Share**       | 21%                   | 28%                       | 14%                   | Not specified        |
| **Products/Services**  | Cloud computing, AI, enterprise software | Extensive cloud services | Data analytics, machine learning | Cloud services, ERP  |
| **Pricing Strategies** | Competitive pricing with various tiers | Generally higher pricing for certain services | Competitive, especially for startups | Competitive, but perceived as complex |
| **Technology/Innovation** | Strong focus on AI integration and strategic alliances | Leading in innovation and service breadth | Strong in data analytics and ML | Comprehensive but slower innovation pace |
| **Supply Chain Capabilities** | Strong partnerships and alliances | Extensive global infrastructure | Growing but less extensive than AWS | Focused on enterprise solutions |
| **Customer Sentiment** | Generally positive, strong enterprise loyalty | High satisfaction but concerns over pricing | Mixed, strong in tech sectors | Positive among enterprise users |

## SWOT Analysis
### Strengths
- Strong brand recognition and loyalty.
- Extensive product offerings across software and cloud services.
- Significant investments in AI and strategic partnerships.

### Weaknesses
- Vulnerability to regulatory scrutiny and economic factors.
- Capacity constraints in AI development.

### Opportunities
- Growth potential in AI integration and cloud services.
- Expansion into emerging markets and new product launches.

### Threats
- Intense competition from AWS, Google Cloud, and Oracle.
- Economic downturns and regulatory challenges.

## Strategic Recommendations
1. **Enhance AI Capabilities**: Continue investing in AI technologies and partnerships to maintain a competitive edge in innovation and product offerings.
2. **Competitive Pricing Strategies**: Review and adjust pricing models to ensure competitiveness against AWS and Google Cloud, particularly for small and medium enterprises.
3. **Expand Market Reach**: Focus on emerging markets and industries that are increasingly adopting cloud solutions, leveraging strategic alliances to penetrate these markets.
4. **Strengthen Customer Engagement**: Enhance customer support and engagement strategies to improve satisfaction and loyalty, particularly among enterprise clients.
5. **Monitor Regulatory Landscape**: Stay proactive in addressing regulatory challenges and ensure compliance to mitigate risks associated with operational constraints.

## Conclusion
Microsoft is well-positioned in the technology sector, particularly in cloud computing and AI. However, it faces significant competition from AWS, Google Cloud, and Oracle. By leveraging its strengths and addressing weaknesses, Microsoft can capitalize on emerging opportunities while navigating potential threats. Implementing the recommended strategies will enhance Microsoft's competitiveness and ensure sustained growth in the evolving technology landscape.

In [31]:

# Verify the OpenAI Chat Model API connection
# Run after initializing the llm object.

try:
    response = llm.invoke(
        "Reply with only the word SUCCESS"
    )

    print("LLM API connection successful!")
    print("Response:", response.content)

except Exception as e:
    print("LLM API connection failed.")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

LLM API connection successful!
Response: SUCCESS


In [32]:

# Execute the complete Multi-Agent RAG workflow
# Test company: Microsoft

# Step 1: Initialize the workflow state
initial_state = CompetitiveAnalysisState(
    company_name="Microsoft",
    max_num_of_questions=5,
    vectorstore=vectorstore
)

# Step 2: Execute the compiled workflow
print("Starting Multi-Agent RAG Workflow...")
print("Company:", initial_state.company_name)

try:
    result = competitive_analysis.invoke(initial_state)

    # Step 3: Display execution results
    print("\nWorkflow execution completed!")
    print("Company:", result.get("company_name"))
    print("Sector:", result.get("sector"))
    print("Valid Company:", result.get("is_valid_company"))
    print("Questions Generated:",
          len(result.get("question_list") or []))
    print("Research Results:",
          len(result.get("qna_results") or []))
    print("ChromaDB Status:",
          result.get("chromadb_insert_status"))
    print("Error Message:", result.get("error_message"))

    # Step 4: Confirm report generation
    if result.get("report"):
        print("\nCompetitive Analysis Report generated successfully!")
        print("Report length:", len(result["report"]))
    else:
        print("\nNo report generated. Check the error message.")

except Exception as e:
    print("\nWorkflow execution failed.")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))


Starting Multi-Agent RAG Workflow...
Company: Microsoft

Workflow execution failed.
Error type: NameError
Error details: name 'competitive_analysis' is not defined


In [33]:

# Display the complete Competitive Analysis Report
# Render the Markdown report in Jupyter Notebook

from IPython.display import display, Markdown

if result.get("report"):
    print("Company:", result.get("company_name"))
    print("Sector:", result.get("sector"))
    print("Report length:", len(result["report"]))
    print("\n--- Complete Competitive Analysis Report ---\n")

    display(Markdown(result["report"]))

else:
    print("No report was generated.")
    print("Error:", result.get("error_message"))


NameError: name 'result' is not defined

# **Executing the Workflow (6 Marks)**

Construct the end-to-end competitive analysis workflow using LangGraph by
connecting all previously defined nodes in a sequential execution pattern.

In this cell, implement logic to:
- Initialize a `StateGraph` using the `CompetitiveAnalysisState`.
- Register each node involved in the workflow, including question generation,
  data retrieval and storage, and report drafting.
- Connect the nodes in sequence to enforce a linear execution flow from
  `START` to `END`.
- Compile the graph to produce an executable workflow.
- Attempt to visualize the compiled workflow using a graph representation,
  while safely handling environments where visualization dependencies are
  unavailable.

This workflow coordinates all pipeline stages and defines the execution order
for the competitive analysis system.

Write your code in the below cell.


In [ ]:

# Build the LangGraph Workflow
# Sequential Design Pattern:
# START -> Question Generator -> Data Retrieval & Storage
# -> Report Drafter -> END

# Step 1: Initialize the StateGraph
workflow = StateGraph(CompetitiveAnalysisState)

# Step 2: Register the three workflow nodes
workflow.add_node(
    "question_generator",
    run_question_generator
)

workflow.add_node(
    "data_retrieval_storage",
    run_data_retrieval_storage
)

workflow.add_node(
    "report_drafter",
    run_report_drafter
)

# Step 3: Define the sequential execution order
workflow.add_edge(START, "question_generator")

workflow.add_edge(
    "question_generator",
    "data_retrieval_storage"
)

workflow.add_edge(
    "data_retrieval_storage",
    "report_drafter"
)

workflow.add_edge("report_drafter", END)

# Step 4: Compile the workflow
competitive_analysis = workflow.compile()

print("LangGraph workflow compiled successfully!")


In [ ]:
# Verify that the LangGraph workflow was compiled

print("Workflow variable exists:",
      "competitive_analysis" in globals())

if "competitive_analysis" in globals():
    print("Workflow type:", type(competitive_analysis).__name__)
else:
    print("ERROR: LangGraph workflow is not initialized.")

In [ ]:

# Visualize the LangGraph workflow
# Display the actual error if visualization fails

try:
    display(
        Image(
            competitive_analysis.get_graph().draw_mermaid_png()
        )
    )
    print("Workflow visualization generated successfully!")

except Exception as e:
    print("Workflow visualization failed.")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))


# **Execution (1 Marks)**

Invoking the workflow with the company  name as input.

In [ ]:

# Execute the complete Multi-Agent RAG workflow
# Company selected for competitive analysis: Microsoft

# Step 1: Initialize the workflow state
initial_state = CompetitiveAnalysisState(
    company_name="Microsoft",
    max_num_of_questions=5,
    vectorstore=vectorstore
)

# Step 2: Execute the compiled LangGraph workflow
print("Starting Multi-Agent RAG Workflow...")
print("Company:", initial_state.company_name)

result = competitive_analysis.invoke(initial_state)

# Step 3: Display execution results
print("\nWorkflow execution completed!")

print("Company:", result.get("company_name"))
print("Sector:", result.get("sector"))
print("Valid Company:", result.get("is_valid_company"))
print("Questions Generated:", len(result.get("question_list") or []))
print("Research Results:", len(result.get("qna_results") or []))
print("ChromaDB Status:", result.get("chromadb_insert_status"))
print("Error Message:", result.get("error_message"))

# Step 4: Confirm report generation
if result.get("report"):
    print("\nCompetitive Analysis Report generated successfully!")
else:
    print("\nNo report generated. Check the error message.")


## Business Insights and Recommendations

### Key Observations

The Multi-Agent RAG Collaborative System successfully generated a competitive analysis report for Microsoft using a sequential workflow.

1. **Automated Research:** The Questions Generator Agent generated five targeted research questions, reducing the need for manual research planning.

2. **External Information Retrieval:** The Data Retrieval and Storage Agent retrieved competitive intelligence using Tavily Search and successfully stored five documents in ChromaDB.

3. **Retrieval-Augmented Generation:** The Report Drafter retrieved ten relevant documents from the vector database and generated a structured competitive analysis report.

4. **Competitive Landscape:** The analysis identifies strong competition in cloud computing and AI, highlighting the importance of continued innovation and differentiation.

5. **Business Opportunities:** The report identifies opportunities for AI investment, industry-specific cloud solutions, strategic partnerships, and improved customer engagement.

### Strategic Recommendations

1. **Expand AI Capabilities:** Continue investing in AI-driven solutions and partnerships to strengthen Microsoft's competitive position.

2. **Develop Industry-Specific Cloud Solutions:** Expand cloud offerings tailored to industries such as healthcare and financial services.

3. **Improve Supply Chain Efficiency:** Use predictive analytics and automation to improve operational efficiency and sustainability.

4. **Strengthen Regulatory Compliance:** Monitor regulatory developments and proactively address compliance risks.

5. **Enhance Customer Engagement:** Use customer feedback and analytics to improve satisfaction, retention, and product differentiation.

### Technical Insights

The project demonstrates how specialized AI agents can collaborate to automate competitor research. LangGraph coordinates sequential execution, Tavily provides external information, ChromaDB enables semantic retrieval, and an LLM synthesizes the retrieved information into an actionable report.

The successful execution validates the end-to-end integration of these components. However, the factual accuracy, freshness, and completeness of the generated analysis should be independently verified against authoritative sources.

### Future Improvements

The solution could be enhanced by incorporating source URLs and publication dates, scheduling periodic competitor research, validating retrieved information against multiple sources, and adding quantitative competitive metrics.

### Conclusion

The Multi-Agent RAG Collaborative System demonstrates the feasibility of automating competitive intelligence research. It reduces manual effort and produces structured business insights that can support strategic decision-making. Additional source validation and quantitative analysis would improve its reliability for production use.

For better readability, displaying the report in Markdown format

In [ ]:
# Display the final Competitive Analysis Report
# in a readable Markdown format

from IPython.display import display, Markdown

if result.get("report"):
    print("Displaying the final Competitive Analysis Report...")
    display(Markdown(result["report"]))
else:
    print("No report was generated.")
    print("Error:", result.get("error_message"))